In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import BertTokenizer
import torch as pt
import re
import numpy as np
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler
from tqdm import tqdm
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from gensim.models import Word2Vec

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# 1. Load and prepare data
def load_texts_for_word2vec(csv_path):
    """Load texts from CSV and tokenize for Word2Vec training"""
    print(f"Loading data from {csv_path}...")
    df = pd.read_csv(csv_path)
    df['combined_text'] = df['combined_text'].astype(str)
    
    # Tokenize texts (simple whitespace tokenization)
    tokenized_texts = [text.lower().split() for text in df['combined_text']]
    labels = df['label'].values
    
    return tokenized_texts, labels, df

# 2. Train Word2Vec model
def train_word2vec(tokenized_texts, vector_size=100, window=5, min_count=2, workers=4, epochs=10):
    """
    Train Word2Vec model
    
    Parameters:
    - vector_size: dimensionality of word vectors (100, 200, 300)
    - window: context window size
    - min_count: ignores words with frequency lower than this
    - workers: number of worker threads
    - epochs: number of training epochs
    """
    print("Training Word2Vec model...")
    model = Word2Vec(
        sentences=tokenized_texts,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=workers,
        epochs=epochs,
        sg=1  # 1 for skip-gram, 0 for CBOW
    )
    print(f"Word2Vec model trained with vocabulary size: {len(model.wv)}")
    return model

# 3. Generate averaged Word2Vec features IN CHUNKS
def get_word2vec_features_chunked(tokenized_texts, w2v_model, labels, output_path, vector_size=100, chunk_size=1000):
    """Convert tokenized texts to averaged Word2Vec vectors and save in chunks"""
    
    # Create column names
    w2v_feature_cols = [f'w2v_{i}' for i in range(vector_size)]
    
    # Write header first
    header_df = pd.DataFrame(columns=w2v_feature_cols + ['label'])
    header_df.to_csv(output_path, index=False)
    
    # Process in chunks
    total_texts = len(tokenized_texts)
    num_chunks = (total_texts + chunk_size - 1) // chunk_size
    
    for chunk_idx in tqdm(range(num_chunks), desc="Processing chunks"):
        start_idx = chunk_idx * chunk_size
        end_idx = min((chunk_idx + 1) * chunk_size, total_texts)
        
        # Process current chunk
        chunk_features = []
        for tokens in tokenized_texts[start_idx:end_idx]:
            # Get vectors for words in vocabulary
            valid_vectors = [w2v_model.wv[word] for word in tokens if word in w2v_model.wv]
            
            if valid_vectors:
                # Average the vectors
                avg_vector = np.mean(valid_vectors, axis=0)
            else:
                # Zero vector if no words found
                avg_vector = np.zeros(vector_size)
            
            chunk_features.append(avg_vector)
        
        # Create DataFrame for this chunk
        chunk_array = np.array(chunk_features)
        chunk_df = pd.DataFrame(chunk_array, columns=w2v_feature_cols)
        chunk_df['label'] = labels[start_idx:end_idx]
        
        # Append to CSV (without header)
        chunk_df.to_csv(output_path, mode='a', header=False, index=False)
        
        # Clear memory
        del chunk_features, chunk_array, chunk_df
    
    print(f"Word2Vec features saved to {output_path}")

# 4. Complete pipeline with chunked saving
def create_word2vec_features(csv_path, output_path, vector_size=100, window=5, chunk_size=1000):
    """Complete pipeline to create Word2Vec features with memory-efficient saving"""
    
    # Load and tokenize
    tokenized_texts, labels, original_df = load_texts_for_word2vec(csv_path)
    
    # Train Word2Vec
    w2v_model = train_word2vec(
        tokenized_texts, 
        vector_size=vector_size, 
        window=window
    )
    
    # Generate and save features in chunks
    get_word2vec_features_chunked(
        tokenized_texts, 
        w2v_model, 
        labels, 
        output_path, 
        vector_size=vector_size,
        chunk_size=chunk_size
    )
    
    # Save the model
    model_path = output_path.replace('.csv', '.model')
    w2v_model.save(model_path)
    print(f"Word2Vec model saved to {model_path}")
    
    # Return model and first few rows for verification
    verification_df = pd.read_csv(output_path, nrows=5)
    
    return verification_df, w2v_model

In [ ]:
w2v_df, w2v_model = create_word2vec_features(
    csv_path='final_datasets/Combined_Train.csv',
    output_path='preprocessed_datasets/word2vec_features.csv',
    vector_size=100,  # 100, 200, or 300
    window=5,
    chunk_size=100
)

print("\nWord2Vec features generated!")
print(f"Shape: {w2v_df.shape}")
w2v_df.head()

Loading data from final_datasets/Combined_Train.csv...
Training Word2Vec model...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

Word2Vec model trained with vocabulary size: 425189


Processing chunks: 100%|██████████| 1801/1801 [02:51<00:00, 10.47it/s]


Word2Vec features saved to preprocessed_datasets/word2vec_features.csv
Word2Vec model saved to preprocessed_datasets/word2vec_features.model

Word2Vec features generated!
Shape: (5, 101)


,w2v_0,w2v_1,w2v_2,w2v_3,w2v_4,w2v_5,w2v_6,w2v_7,w2v_8,w2v_9,...,w2v_91,w2v_92,w2v_93,w2v_94,w2v_95,w2v_96,w2v_97,w2v_98,w2v_99,label
0,-0.088032,0.216574,0.071848,-0.047545,0.058198,-0.256617,0.260833,0.349325,-0.071233,0.019895,...,0.007162,0.108326,-0.048984,0.199117,0.193159,0.010972,-0.129852,-0.008635,-0.025235,1
1,-0.105712,0.164476,0.113690,-0.072790,0.047659,-0.264727,0.198774,0.418022,-0.094503,-0.068364,...,-0.031178,0.161627,0.003181,0.201607,0.181485,-0.037429,-0.178130,-0.130464,-0.065710,0
2,-0.052271,0.156961,0.104463,-0.031137,-0.004413,-0.282800,0.173517,0.376382,-0.055175,-0.168959,...,0.014802,0.117292,-0.042914,0.229592,0.126582,-0.080868,-0.130012,-0.070988,-0.104734,1
3,-0.001157,0.231302,0.040211,-0.092716,0.109829,-0.264168,0.174594,0.458767,-0.087711,-0.222695,...,0.062020,0.120198,-0.043880,0.228383,0.158327,-0.116205,-0.136470,-0.038554,-0.114638,0
4,-0.081945,0.190545,0.036891,-0.018873,-0.040791,-0.260958,0.199944,0.391738,-0.177649,-0.122187,...,0.070465,0.195436,-0.049793,0.194982,0.188618,-0.036434,-0.102704,-0.022445,-0.066967,1


In [ ]:
def scale_data(dataframe, oversample=False):
  x = dataframe[dataframe.columns[:-1]].values
  y = dataframe[dataframe.columns[-1]].values

  scaler = StandardScaler()
  x = scaler.fit_transform(x)

  if oversample:
    ros = RandomOverSampler()
    x, y = ros.fit_resample(x, y)

  data = np.hstack((x, np.reshape(y, (-1, 1))))

  return data, x, y

In [ ]:
def split_dataset(df):
    train = df.sample(frac=0.8, random_state=42)
    test = df.drop(train.index)
    print(f"Total rows in dataset: {len(df)}")
    print(f"Total rows in train set: {len(train)}")
    print(f"Total rows in test set: {len(test)}")

    print("\nClass distribution in full dataset:")
    print(df['label'].value_counts())

    print("\nClass distribution in train set:")
    print(train['label'].value_counts())

    print("\nClass distribution in test set:")
    print(test['label'].value_counts())
    
    train, xtrain, ytrain = scale_data(train)
    test, xtest, ytest = scale_data(test)
    
    print("\n KNN \n")

    return train, xtrain, ytrain, test, xtest, ytest

In [ ]:
def train_ml_models(xtrain, ytrain, xtest, ytest):
    print("\n KNN \n")
    knn_model = KNeighborsClassifier(n_neighbors=5)
    knn_model.fit(xtrain, ytrain)

    ypred = knn_model.predict(xtest)
    print(classification_report(ytest, ypred))
    
    print("\n Gaussian NB \n")
    nb_model = GaussianNB()
    nb_model = nb_model.fit(xtrain, ytrain)

    ypred = nb_model.predict(xtest)

    print(classification_report(ytest, ypred))
    
    print("\n Logistica Regression \n")
    lgr_model = LogisticRegression()
    lgr_model = lgr_model.fit(xtrain, ytrain)

    ypred = lgr_model.predict(xtest)

    print(classification_report(ytest, ypred))

    print("\n Support Vector Classifier \n")
    svc_model = SVC()
    svc_model = svc_model.fit(xtrain, ytrain)

    ypred = svc_model.predict(xtest)
    print(classification_report(ytest, ypred))
    
    print("\n XGBoost \n")
    xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
    xgb_model = xgb_model.fit(xtrain, ytrain)

    ypred = xgb_model.predict(xtest)
    print(classification_report(ytest, ypred))
    
    print("\n Random Forest Classifier \n")
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_model.fit(xtrain, ytrain)

    ypred = rf_model.predict(xtest)
    print(classification_report(ytest, ypred))

[0 1 1 ... 0 1 0]
[1 1 1 ... 0 1 0]
              precision    recall  f1-score   support

           0       0.81      0.91      0.86      6982
           1       0.90      0.80      0.85      7443

    accuracy                           0.85     14425
   macro avg       0.86      0.85      0.85     14425
weighted avg       0.86      0.85      0.85     14425

